In [3]:
import os
import time
import json
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ---------- CONFIG ----------
BASE_URL = "https://developer.ibm.com/apis/catalog/instana--instana-rest-api/api/API--instana--instana-rest-api-documentation"
DOMAIN_PREFIX = "https://developer.ibm.com/apis/catalog/instana--instana-rest-api/api/"

# Sidebar navigation selector
SIDEBAR_SELECTOR = "#root > div > div.overview-page.hide-scrollbar.grid-width > div.flex-grid.cds--grid > div.custom-grid-1 > div > div.side-nav-column.pr-0.cds--col-sm-4.cds--col-md-2.cds--col-lg-3.cds--col-xlg-3.cds--col-max-3 > div.side-nav > nav > ul > div > li > ul"

# ---------- SETUP ----------
options = webdriver.ChromeOptions()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
driver = webdriver.Chrome(options=options)

# ---------- HELPERS ----------

def extract_sidebar_links(driver, base_url):
    endpoints = []

    wait = WebDriverWait(driver, 10)
    sidebar = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, SIDEBAR_SELECTOR)))
    divs = sidebar.find_elements(By.XPATH, "./div")

    for div in divs:
        try:
            nested_uls = div.find_elements(By.XPATH, ".//ul")
            for ul in nested_uls:
                anchors = ul.find_elements(By.TAG_NAME, "a")
                for anchor in anchors:
                    href = anchor.get_attribute("href")
                    if href and "#" in href:
                        endpoint = href.split("#")[-1]
                        if endpoint:
                            endpoints.append(endpoint)
        except Exception as e:
            continue
    
    return endpoints

# ---------- MAIN ----------

try:
    print(f"Visiting base URL: {BASE_URL}")
    driver.get(BASE_URL)
    time.sleep(3)

    extracted_endpoints = extract_sidebar_links(driver, BASE_URL)
    extracted_endpoints = sorted(set(extracted_endpoints))  # remove duplicates + sort

    print("\n✅ Extracted Endpoints:")
    print(json.dumps(extracted_endpoints, indent=2))

except Exception as e:
    print(f"[CRITICAL ERROR]: {e}")

finally:
    driver.quit()


Visiting base URL: https://developer.ibm.com/apis/catalog/instana--instana-rest-api/api/API--instana--instana-rest-api-documentation

✅ Extracted Endpoints:
[
  "addActionInstance",
  "addApplicationConfig",
  "addCustomDashboard",
  "addManualServiceConfig",
  "addPermissionsOnGroup",
  "addPolicies",
  "addPolicy",
  "addServiceConfig",
  "addUsersToGroup",
  "agentMonitoringEvents",
  "clearMobileAppSourceMapUploadConfiguration",
  "clearSourceMapUploadConfiguration",
  "commitMobileAppSourceMapFile",
  "createApdexConfiguration",
  "createApplicationAlertConfig",
  "createBusinessPerspective",
  "createCustomEntities",
  "createEndpointConfig",
  "createGlobalApplicationAlertConfig",
  "createGroup",
  "createGroupMapping",
  "createHttpEndpointConfig",
  "createInfraAlertConfig",
  "createLogAlertConfig",
  "createMobileAppAlertConfig",
  "createServiceLevelsAlertConfig",
  "createSliConfig",
  "createSliConfigV2",
  "createSloConfig",
  "createSloCorrectionWindowConfig",
  "creat

In [2]:
import os
import time
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

# ---------- CONFIG ----------
prefix = "https://developer.ibm.com/apis/catalog/instana--instana-rest-api/api/API--instana--instana-rest-api-documentation#"
suffixes = ["addActionInstance", "addApplicationConfig", "addCustomDashboard"]

# suffixes = [
#   "addActionInstance",
#   "addApplicationConfig",
#   "addCustomDashboard",
#   "addManualServiceConfig",
#   "addPermissionsOnGroup",
#   "addPolicies",
#   "addPolicy",
#   "addServiceConfig",
#   "addUsersToGroup",
#   "agentMonitoringEvents",
#   "clearMobileAppSourceMapUploadConfiguration",
#   "clearSourceMapUploadConfiguration",
#   "commitMobileAppSourceMapFile",
#   "createApdexConfiguration",
#   "createApplicationAlertConfig",
#   "createBusinessPerspective",
#   "createCustomEntities",
#   "createEndpointConfig",
#   "createGlobalApplicationAlertConfig",
#   "createGroup",
#   "createGroupMapping",
#   "createHttpEndpointConfig",
#   "createInfraAlertConfig",
#   "createLogAlertConfig",
#   "createMobileAppAlertConfig",
#   "createServiceLevelsAlertConfig",
#   "createSliConfig",
#   "createSliConfigV2",
#   "createSloConfig",
#   "createSloCorrectionWindowConfig",
#   "createSyntheticAlertConfig",
#   "createSyntheticCredential",
#   "createSyntheticTest",
#   "createSyntheticTestCICD",
#   "createWebsite",
#   "createWebsiteAlertConfig",
#   "deleteActionInstance",
#   "deleteActionInstances",
#   "deleteAlert",
#   "deleteAlertingChannel",
#   "deleteApdexConfiguration",
#   "deleteApiToken",
#   "deleteApplicationAlertConfig",
#   "deleteApplicationConfig",
#   "deleteBuiltInEventSpecification",
#   "deleteBusinessPerspective",
#   "deleteCustomDashboard",
#   "deleteCustomEntity",
#   "deleteCustomEventSpecification",
#   "deleteCustomPayloadConfiguration",
#   "deleteEndpointConfig",
#   "deleteGlobalApplicationAlertConfig",
#   "deleteGoogleSSOConfig",
#   "deleteGroup",
#   "deleteGroupMapping",
#   "deleteGroupMappings",
#   "deleteGroups",
#   "deleteHttpEndpointConfig",
#   "deleteInfraAlertConfig",
#   "deleteLdapConfig",
#   "deleteLogAlertConfig",
#   "deleteMaintenanceConfig",
#   "deleteMaintenanceConfigV2",
#   "deleteManualServiceConfig",
#   "deleteMobileAppAlertConfig",
#   "deleteMobileAppConfig",
#   "deleteMobileAppSourceMapUploadConfiguration",
#   "deleteOidcConfig",
#   "deletePolicy",
#   "deleteRelease",
#   "deleteSamlConfig",
#   "deleteServiceConfig",
#   "deleteServiceLevelsAlertConfig",
#   "deleteSessionSettings",
#   "deleteSliConfig",
#   "deleteSliConfigV2",
#   "deleteSloConfig",
#   "deleteSloCorrectionWindowConfig",
#   "deleteSyntheticAlertConfig",
#   "deleteSyntheticCall",
#   "deleteSyntheticCredential",
#   "deleteSyntheticLocation",
#   "deleteSyntheticTest",
#   "deleteTeams",
#   "deleteWebsite",
#   "deleteWebsiteAlertConfig",
#   "deleteWebsiteSourceMapUploadConfiguration",
#   "disableApplicationAlertConfig",
#   "disableBuiltInEventSpecification",
#   "disableCustomEventSpecification",
#   "disableGlobalApplicationAlertConfig",
#   "disableInfraAlertConfig",
#   "disableLogAlertConfig",
#   "disableMobileAppAlertConfig",
#   "disableServiceLevelsAlertConfig",
#   "disableSyntheticAlertConfig",
#   "disableWebsiteAlertConfig",
#   "enableApplicationAlertConfig",
#   "enableBuiltInEventSpecification",
#   "enableCustomEventSpecification",
#   "enableGlobalApplicationAlertConfig",
#   "enableInfraAlertConfig",
#   "enableLogAlertConfig",
#   "enableMobileAppAlertConfig",
#   "enableServiceLevelsAlertConfig",
#   "enableSyntheticAlertConfig",
#   "enableWebsiteAlertConfig",
#   "findActiveApplicationAlertConfigs",
#   "findActiveGlobalApplicationAlertConfigs",
#   "findActiveInfraAlertConfigs",
#   "findActiveLogAlertConfigs",
#   "findActiveMobileAppAlertConfigs",
#   "findActiveServiceLevelsAlertConfigs",
#   "findActiveSyntheticAlertConfigs",
#   "findActiveWebsiteAlertConfigs",
#   "findApplicationAlertConfig",
#   "findApplicationAlertConfigVersions",
#   "findGlobalApplicationAlertConfig",
#   "findGlobalApplicationAlertConfigVersions",
#   "findInfraAlertConfig",
#   "findInfraAlertConfigVersions",
#   "findLogAlertConfig",
#   "findLogAlertConfigVersions",
#   "findMobileAppAlertConfig",
#   "findMobileAppAlertConfigVersions",
#   "findServiceLevelsAlertConfig",
#   "findServiceLevelsAlertConfigVersions",
#   "findSyntheticAlertConfig",
#   "findSyntheticAlertConfigVersions",
#   "findWebsiteAlertConfig",
#   "findWebsiteAlertConfigVersions",
#   "getAccessLogs",
#   "getActionByID",
#   "getActionInstance",
#   "getActionInstances",
#   "getActionMatches",
#   "getActions",
#   "getActivities",
#   "getActivitiesCsv",
#   "getAgentClrLogs",
#   "getAgentLogs",
#   "getAgentSnapshot",
#   "getAgentSupportInformation",
#   "getAlert",
#   "getAlertingChannel",
#   "getAlertingChannels",
#   "getAlertingChannelsOverview",
#   "getAlertingConfigurationInfos",
#   "getAlerts",
#   "getAllApdexConfigurations",
#   "getAllManualServiceConfigs",
#   "getAllMobileAppCatalogTags",
#   "getAllReleases",
#   "getAllSliConfigs",
#   "getAllSliConfigsV2",
#   "getAllSloConfigTags",
#   "getAllSloConfigs",
#   "getAllSloCorrectionWindowConfigs",
#   "getAllUsage",
#   "getApdexConfiguration",
#   "getApdexConfigurationsForEntityTypeAndId",
#   "getApdexReport",
#   "getApiToken",
#   "getApiTokens",
#   "getApplicationCatalogMetrics",
#   "getApplicationConfig",
#   "getApplicationConfigs",
#   "getApplicationDataMetricsV2",
#   "getApplicationEndpoints",
#   "getApplicationMetrics",
#   "getApplicationServices",
#   "getApplicationTagCatalog",
#   "getApplicationTags",
#   "getApplications",
#   "getAuditLogs",
#   "getAvailableMetrics",
#   "getAvailablePayloadKeysByPluginId",
#   "getAvailablePlugins",
#   "getBeaconGroups",
#   "getBeaconMetrics",
#   "getBeaconMetricsV2",
#   "getBeacons",
#   "getBuiltInEventSpecification",
#   "getBuiltInEventSpecifications",
#   "getBusinessPerspective",
#   "getBusinessPerspectives",
#   "getBusinessTagCatalog",
#   "getCallDetails",
#   "getCallGroup",
#   "getCorrelatedTraces",
#   "getCustomDashboard",
#   "getCustomDashboards",
#   "getCustomEntity",
#   "getCustomEventSpecification",
#   "getCustomEventSpecifications",
#   "getCustomPayloadConfigurations",
#   "getCustomPayloadTagCatalog",
#   "getDynamicParametersTagCatalog",
#   "getEndpointConfig",
#   "getEndpointConfigs",
#   "getEndpointsMetrics",
#   "getEntities",
#   "getEntityGroups",
#   "getEvent",
#   "getEventSpecificationInfos",
#   "getEventSpecificationInfosByIds",
#   "getEvents",
#   "getEventsByIds",
#   "getGroup",
#   "getGroupMappings",
#   "getGroups",
#   "getGroupsByUser",
#   "getHealthState",
#   "getHostsPerDay",
#   "getHostsPerMonth",
#   "getHttpEndpointConfig",
#   "getHttpEndpointConfigs",
#   "getIdentityProviderPatch",
#   "getInfrastructureCatalogMetrics",
#   "getInfrastructureCatalogPlugins",
#   "getInfrastructureCatalogPluginsWithCustomMetrics",
#   "getInfrastructureCatalogSearchFields",
#   "getInfrastructureMetrics",
#   "getInvitations",
#   "getLocationSummaryList",
#   "getMaintenanceConfig",
#   "getMaintenanceConfigV2",
#   "getMaintenanceConfigs",
#   "getMaintenanceConfigsV2",
#   "getMetricsResult",
#   "getMobileAppBeaconGroups",
#   "getMobileAppBeaconMetrics",
#   "getMobileAppBeaconMetricsV2",
#   "getMobileAppBeacons",
#   "getMobileAppConfig",
#   "getMobileAppGeoLocationConfiguration",
#   "getMobileAppGeoMappingRules",
#   "getMobileAppIpMaskingConfiguration",
#   "getMobileAppMetricCatalog",
#   "getMobileAppSourceMapFile",
#   "getMobileAppSourceMapFiles",
#   "getMobileAppTagCatalog",
#   "getMonitoringState",
#   "getOneSyntheticCredentialAssociations",
#   "getPageLoad",
#   "getPluginPayload",
#   "getPolicies",
#   "getPolicyByID",
#   "getRelatedHosts",
#   "getRelease",
#   "getServiceConfig",
#   "getServiceConfigs",
#   "getServices",
#   "getServicesMap",
#   "getServicesMetrics",
#   "getSession",
#   "getSessionSettings",
#   "getShareableApiTokens",
#   "getShareableUsers",
#   "getSli",
#   "getSliConfig",
#   "getSliConfigV2",
#   "getSliConfigsForEntityTypeAndIdV2",
#   "getSlo",
#   "getSloConfigById",
#   "getSloCorrection",
#   "getSloCorrectionWindowConfigById",
#   "getSnapshot",
#   "getSnapshots",
#   "getSyntheticCalls",
#   "getSyntheticCatalogMetrics",
#   "getSyntheticCredentialAssociations",
#   "getSyntheticCredentialNames",
#   "getSyntheticDatacenter",
#   "getSyntheticDatacenters",
#   "getSyntheticLocation",
#   "getSyntheticLocations",
#   "getSyntheticResult",
#   "getSyntheticResultDetailData",
#   "getSyntheticResultDetailDataFile",
#   "getSyntheticResultList",
#   "getSyntheticResultMetadata",
#   "getSyntheticTagCatalog",
#   "getSyntheticTest",
#   "getSyntheticTestCICD",
#   "getSyntheticTestCICDs",
#   "getSyntheticTests",
#   "getSystemRules",
#   "getTagCatalog",
#   "getTagCatalogAll",
#   "getTestSummaryList",
#   "getTopology",
#   "getTrace",
#   "getTraceDownload",
#   "getTraceGroups",
#   "getTraces",
#   "getUsagePerDay",
#   "getUsagePerMonth",
#   "getUserById",
#   "getUsers",
#   "getUsersIncludingInvitations",
#   "getVersion",
#   "getWebsite",
#   "getWebsiteCatalogMetrics",
#   "getWebsiteCatalogTags",
#   "getWebsiteGeoLocationConfiguration",
#   "getWebsiteGeoMappingRules",
#   "getWebsiteIpMaskingConfiguration",
#   "getWebsiteSourceMapUploadConfiguration",
#   "getWebsiteSourceMapUploadConfigurations",
#   "getWebsiteTagCatalog",
#   "getWebsites",
#   "inviteUsers",
#   "kubernetesInfoEvents",
#   "listCustomEntities",
#   "listEntitiesFromInfraEntities",
#   "manuallyCloseEvent",
#   "multiCloseEvent",
#   "orderServiceConfig",
#   "patchSyntheticCredential",
#   "patchSyntheticTest",
#   "pauseRecurrentMaintenanceConfiguration",
#   "postAlertingChannel",
#   "postApiToken",
#   "postCustomEventSpecification",
#   "postMobileAppConfig",
#   "postMobileAppSourceMapConfig",
#   "postRelease",
#   "postSnapshots",
#   "postWebsiteSourceMapUploadConfig",
#   "putAlert",
#   "putAlertingChannel",
#   "putApiToken",
#   "putApplicationConfig",
#   "putCustomEventSpecification",
#   "putMaintenanceConfig",
#   "putMaintenanceConfigV2",
#   "putRelease",
#   "putServiceConfig",
#   "removeUserFromGroup",
#   "removeUserFromTenant",
#   "removeUsersFromTenant",
#   "renameMobileAppConfig",
#   "renameWebsite",
#   "replaceAll",
#   "replaceAllManualServiceConfigs",
#   "resolve",
#   "restoreApplicationAlertConfig",
#   "restoreGlobalApplicationAlertConfig",
#   "restoreInfraAlertConfig",
#   "restoreLogAlertConfig",
#   "restoreMobileAppAlertConfig",
#   "restoreServiceLevelsAlertConfig",
#   "restoreSyntheticAlertConfig",
#   "restoreWebsiteAlertConfig",
#   "resumeRecurrentMaintenanceConfiguration",
#   "revokePendingInvitation",
#   "searchHostAgents",
#   "sendTestAlerting",
#   "sendTestAlertingById",
#   "setMobileAppGeoMappingRules",
#   "setSessionSettings",
#   "setWebsiteGeoMappingRules",
#   "shareAndInviteUsers",
#   "softwareVersions",
#   "updateAgent",
#   "updateApplicationAlertConfig",
#   "updateApplicationHistoricBaseline",
#   "updateBusinessPerspective",
#   "updateConfigurationByHost",
#   "updateConfigurationByQuery",
#   "updateCustomDashboard",
#   "updateCustomEntity",
#   "updateEndpointConfig",
#   "updateGlobalApplicationAlertConfig",
#   "updateGroup",
#   "updateGroupMapping",
#   "updateHttpEndpointConfig",
#   "updateIdentityProvider",
#   "updateInfraAlertConfig",
#   "updateLogAlertConfig",
#   "updateManualServiceConfig",
#   "updateMobileAppAlertConfig",
#   "updateMobileAppGeoLocationConfiguration",
#   "updateMobileAppHistoricBaseline",
#   "updateMobileAppIpMaskingConfiguration",
#   "updatePolicy",
#   "updateServiceLevelsAlertConfig",
#   "updateSliConfig",
#   "updateSloConfig",
#   "updateSloCorrectionWindowConfig",
#   "updateSyntheticAlertConfig",
#   "updateSyntheticCall",
#   "updateSyntheticCredential",
#   "updateSyntheticTest",
#   "updateUser",
#   "updateWebsiteAlertConfig",
#   "updateWebsiteGeoLocationConfiguration",
#   "updateWebsiteHistoricBaseline",
#   "updateWebsiteIpMaskingConfiguration",
#   "uploadMobileAppSourceMapFile",
#   "uploadSourceMapFile",
#   "upsertCustomPayloadConfiguration",
#   "upsertCustomPayloadConfigurationV2"
# ]

DEBUG = False  # Set to False to store in ChromaDB

chroma_path = "/data/wikidata/chromadb" # "concert_docs/chroma_db" 
collection_name = "instana_api"
embedding_fn = SentenceTransformerEmbeddingFunction("all-MiniLM-L6-v2")

# ---------- SETUP ----------
if not DEBUG:
    client = chromadb.PersistentClient(path=chroma_path)
    collection = client.get_or_create_collection(
        name=collection_name,
        embedding_function=embedding_fn
    )

# Initialize Chrome
options = webdriver.ChromeOptions()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
driver = webdriver.Chrome(options=options)

def wait_for_main_content(driver):
    """Wait for the main content container to load."""
    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.XPATH, "/html/body/div[1]/div/div[3]/div[2]/div[2]/div/div[2]/div/div/div[1]/div"))
        )
        time.sleep(2)
        return True
    except Exception:
        return False

def extract_main_content(driver):
    """Extract main page content."""
    try:
        container = driver.find_element(By.XPATH, "/html/body/div[1]/div/div[3]/div[2]/div[2]/div/div[2]/div/div/div[1]/div")
        html_content = container.get_attribute("innerHTML")
        clean_text = re.sub(r'<[^>]+>', ' ', html_content)
        clean_text = ' '.join(clean_text.split()).strip()
        return clean_text
    except Exception:
        return None

def find_and_expand_arrows(driver, container, max_depth=10, current_depth=0):
    """Recursively find and expand arrow elements to extract hidden content."""
    if current_depth >= max_depth:
        return []
    
    extracted_contents = []
    
    try:
        arrow_selector = ".//div[contains(@class, 'arrow') and contains(@class, 'arrow-text') and not(contains(@class, 'arrow-down'))]"
        arrows = container.find_elements(By.XPATH, arrow_selector)
        
        for arrow in arrows:
            try:
                current_classes = arrow.get_attribute("class")
                if "arrow-down" in current_classes:
                    continue
                
                arrow_text = arrow.text.strip()
                
                # Find parent element
                try:
                    parent_element = arrow.find_element(By.XPATH, "./ancestor::li[1]")
                except NoSuchElementException:
                    try:
                        parent_element = arrow.find_element(By.XPATH, "./ancestor::div[contains(@class, 'parameter')][1]")
                    except NoSuchElementException:
                        parent_element = arrow.find_element(By.XPATH, "./..")
                
                initial_text_length = len(parent_element.text)
                
                # Click arrow
                driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", arrow)
                time.sleep(0.5)
                driver.execute_script("arguments[0].click();", arrow)
                time.sleep(1.5)
                
                # Check if expanded
                updated_classes = arrow.get_attribute("class")
                if "arrow-down" in updated_classes:
                    updated_text_length = len(parent_element.text)
                    content_growth = updated_text_length - initial_text_length
                    
                    if content_growth > 10:
                        updated_content = parent_element.get_attribute("innerHTML")
                        clean_content = re.sub(r'<[^>]+>', ' ', updated_content)
                        clean_content = ' '.join(clean_content.split()).strip()
                        
                        content_info = {
                            'arrow_text': arrow_text,
                            'content': clean_content
                        }
                        extracted_contents.append(content_info)
                    
                    # Recursively process nested arrows
                    time.sleep(1)
                    nested_contents = find_and_expand_arrows(driver, parent_element, max_depth, current_depth + 1)
                    extracted_contents.extend(nested_contents)
                
            except Exception:
                continue
                
    except Exception:
        pass
    
    return extracted_contents

def find_target_section(driver):
    """Find the specific section containing expandable arrows."""
    try:
        return driver.find_element(By.XPATH, "/html/body/div[1]/div/div[3]/div[2]/div[2]/div/div[2]/div/div/div[1]/div[1]/div[2]/ul")
    except NoSuchElementException:
        return None

# ---------- MAIN PROCESSING ----------

for i, suffix in enumerate(suffixes):
    url = prefix + suffix
    print(f"\nProcessing: {url} ({suffix})")
    
    try:
        # Navigate to page
        driver.get("about:blank")
        driver.get(url)
        time.sleep(2)
        
        if not wait_for_main_content(driver):
            continue
        
        # Extract main content
        main_content = extract_main_content(driver)
        if not main_content:
            continue
        
        # Extract arrow-expanded content
        expanded_content = []
        target_section = find_target_section(driver)
        if target_section:
            expanded_content = find_and_expand_arrows(driver, target_section)
        
        # Combine all content
        combined_content = f"MAIN CONTENT:\n{main_content}\n\n"
        
        if expanded_content:
            combined_content += "EXPANDED CONTENT:\n"
            for content_info in expanded_content:
                combined_content += f"\n--- {content_info['arrow_text']} ---\n"
                combined_content += content_info['content'] + "\n"
        
        # Output extracted content
        print("Extracted content:")
        print(combined_content)
        print("-" * 80)
        
        # Store in ChromaDB or debug
        if DEBUG:
            # Just print for debugging
            pass
        else:
            chunk_id = f"{i}_{suffix}"
            collection.upsert(
                documents=[combined_content],
                ids=[chunk_id],
                metadatas=[{
                    "source": url,
                    "endpoint": suffix
                }]
            )
        
    except Exception as e:
        print(f"Error processing {url}: {e}")

# Clean up
driver.quit()

if not DEBUG:
    print(f"\n✅ Indexed {len(suffixes)} API endpoints into ChromaDB.")
else:
    print(f"\n✅ Debug mode: Processed {len(suffixes)} API endpoints")


/home/gunjan/TheToolkit/schema-mapping-toolkit/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The chromedriver version (137.0.7151.70) detected in PATH at /home/gunjan/bin/chromedriver might not be compatible with the detected chrome version (138.0.7204.49); currently, chromedriver 138.0.7204.49 is recommended for chrome 138.*, so it is advised to delete the driver in PATH and retry



Processing: https://developer.ibm.com/apis/catalog/instana--instana-rest-api/api/API--instana--instana-rest-api-documentation#addActionInstance (addActionInstance)
Extracted content:
MAIN CONTENT:
API reference Expand Run an automation action. POST /api/automation/actioninstances Submits the automation action for execution on an agent. The automation action to execute and the agent on which to execute the action must be specified as actionId and hostId. For more details on the request payload see the request sample. This call requires Execution of automation actions permission. Request URL https://{unit}-{tenant}.instana.io/api/automation/actioninstances Copy to clipboard https://{domain}/api/automation/actioninstances Copy to clipboard Security apiKey authorization required in header In order to use the REST API, you need to create an API token and assign permissions to it. API tokens are managed only by the account owner role, who can create multiple API tokens with varying permissi

In [4]:
## TEST THE CREATED CHROMADB 

import chromadb
from chromadb.config import Settings

chroma_path = "/data/wikidata/chromadb" 
collection_name = "instana_api"
client = chromadb.PersistentClient(path=chroma_path)
collection = client.get_collection(name=collection_name)
collections = client.list_collections()
print("\n🗂 Collections in ChromaDB:")
for col in collections:
    print(f"- {col.name}")

# Load your specific collection (e.g., "turbonomic")
collection = client.get_collection("instana_api") 

# TO PRINT ALL ENTRIES
print(f"\n📦 Collection '{collection.name}' has {collection.count()} documents.\n")
results = collection.get()
print(results)

# TO DELETE A SPECIFIC ENTRY
# bad_id = "032e0fd1d8031a86ec7298652bc62e6b"
# # Delete the entry
# collection.delete(ids=[bad_id])
# print(f"✅ Deleted entry with ID: {bad_id}")

# TO DELETE THE COLLECTION
#client.delete_collection(name="instana_api")


🗂 Collections in ChromaDB:
- instana_api
- test_collection
- concert_api

📦 Collection 'instana_api' has 404 documents.

{'ids': ['0_addActionInstance', '1_addApplicationConfig', '2_addCustomDashboard', '3_addManualServiceConfig', '4_addPermissionsOnGroup', '5_addPolicies', '6_addPolicy', '7_addServiceConfig', '8_addUsersToGroup', '9_agentMonitoringEvents', '10_clearMobileAppSourceMapUploadConfiguration', '11_clearSourceMapUploadConfiguration', '12_commitMobileAppSourceMapFile', '13_createApdexConfiguration', '14_createApplicationAlertConfig', '15_createBusinessPerspective', '16_createCustomEntities', '17_createEndpointConfig', '18_createGlobalApplicationAlertConfig', '19_createGroup', '20_createGroupMapping', '21_createHttpEndpointConfig', '22_createInfraAlertConfig', '23_createLogAlertConfig', '24_createMobileAppAlertConfig', '25_createServiceLevelsAlertConfig', '26_createSliConfig', '27_createSliConfigV2', '28_createSloConfig', '29_createSloCorrectionWindowConfig', '30_createSynthe

In [9]:
# TEST THE RETRIEVAL

results = collection.query(
    query_texts=["ActionInstance"],
    n_results=3,
    include=["documents", "metadatas"]
)

for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print("\n🔎 Matched Doc:\n", doc)
    print("Metadata:", meta)



🔎 Matched Doc:
 MAIN CONTENT:
API reference Expand Run an automation action. POST /api/automation/actioninstances Submits the automation action for execution on an agent. The automation action to execute and the agent on which to execute the action must be specified as actionId and hostId. For more details on the request payload see the request sample. This call requires Execution of automation actions permission. Request URL https://{unit}-{tenant}.instana.io/api/automation/actioninstances Copy to clipboard https://{domain}/api/automation/actioninstances Copy to clipboard Security apiKey authorization required in header In order to use the REST API, you need to create an API token and assign permissions to it. API tokens are managed only by the account owner role, who can create multiple API tokens with varying permissions. API token management can found in Settings → Access Control → API Tokens. Parameters Name Description Content-type string (header) application/json Accept string 